# Persiapan Data Oleh-Oleh

Notebook ini mencatat langkah awal untuk data oleh-oleh. Untuk sekarang, fokusnya baru menyiapkan kategori, aturan kolom, dan rencana pencarian tempat.

## Tahap A - Posisi Data Oleh-Oleh

Oleh-oleh bukan data utama seperti wisata. Data ini dipakai sebagai pelengkap, misalnya saat user ingin mencari tempat membeli oleh-oleh setelah menentukan tujuan jalan-jalan.

**Catatan:** data oleh-oleh dipisahkan dari wisata dan penginapan supaya prosesnya tidak tercampur.

## Tahap B - Cek Kategori Awal

Cell berikut membaca daftar kategori yang sudah dibuat. Ini hanya cek awal sebelum mulai mengambil data dari scraper.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "02_Notebooks":
    WORKSPACE = ROOT.parent
else:
    WORKSPACE = ROOT / "OlehOleh_Workspace"

taxonomy_path = WORKSPACE / "03_Curated" / "OLEH_OLEH_TAXONOMY_2026-06-09.csv"
taxonomy = pd.read_csv(taxonomy_path)

summary = {
    "taxonomy_rows": len(taxonomy),
    "parent_groups": taxonomy["parent_group"].nunique(),
    "high_priority_subtypes": int((taxonomy["mvp_priority"] == "high").sum()),
}
summary

**Catatan:** kategori ini dipakai sebagai pegangan awal. Kalau nanti hasil scraper terlalu melebar, kategorinya bisa dirapikan lagi.

## Tahap C - Batas Awal Pencarian Tempat

Batch pertama dibuat kecil dulu. Fokusnya pada kategori yang paling jelas sebagai oleh-oleh. Fashion, mall umum, dan pasar lokal belum diprioritaskan.

In [ ]:
mvp_taxonomy = taxonomy.loc[
    taxonomy["mvp_priority"].isin(["high", "medium"]),
    ["subtype", "subtype_label", "parent_group", "mvp_priority", "scoring_notes"],
].sort_values(["mvp_priority", "subtype"])

mvp_taxonomy.head(20)

**Catatan:** untuk awal, pencarian diarahkan ke makanan oleh-oleh dan pusat oleh-oleh. Area ini lebih mudah dicek dan tidak terlalu melebar.

## Tahap D - Data Yang Dicari Saat Pencarian Tempat

| Data | Alasan |
|---|---|
| Nama tempat | Nama toko/tempat yang ditemukan |
| Alamat | Membantu cek lokasi dan region |
| Koordinat | Dipakai untuk menghitung jarak dari wisata |
| Rating | Gambaran kualitas awal |
| Jumlah review | Membantu melihat seberapa kuat datanya |
| Kategori Google | Membantu menentukan jenis toko/tempat |
| Google Maps URL / place URL | Dipakai kalau nanti lanjut ambil review |
| Jam buka | Dipakai untuk filter buka sekarang |
| Website / phone | Sumber tambahan jika tersedia |

**Catatan:** tahap ini hanya mencari daftar tempat. Review komentar belum diambil karena daftar tempatnya harus dicek dulu.

## Tahap E - Rencana Query Batch Kecil

Query awal dibuat sedikit dulu supaya hasilnya mudah dicek. Tujuannya bukan mengejar banyak data, tetapi melihat apakah kategori dan kolom yang disiapkan sudah cocok.

In [ ]:
seed_queries = [
    "brownies oleh-oleh Bandung",
    "toko kue oleh-oleh Bandung",
    "bolu oleh-oleh Bandung",
    "snack oleh-oleh Bandung",
    "keripik oleh-oleh Bandung",
    "batagor oleh-oleh Bandung",
    "siomay oleh-oleh Bandung",
    "tahu susu oleh-oleh Lembang",
    "pusat oleh-oleh Bandung",
    "toko oleh-oleh khas Bandung",
    "oleh-oleh Lembang Bandung",
    "oleh-oleh Ciwidey Bandung",
]

pd.DataFrame({"query": seed_queries, "purpose": "place_discovery"})

**Catatan:** query dibuat spesifik memakai kata `oleh-oleh`, `toko`, `pusat`, atau nama produk. Query umum seperti `kuliner Bandung` belum dipakai karena bisa menarik restoran biasa.

## Tahap F - Output Berikutnya

Output yang diharapkan setelah scraping kecil:

| Output | Fungsi |
|---|---|
| Raw place discovery | Data asli dari scraper |
| Candidate setelah dedupe | Data setelah toko/cabang duplikat dikurangi |
| Label kategori awal | Kategori sementara berdasarkan aturan awal |
| Manual review queue | Tempat yang masih ragu |
| Review target ready | Tempat yang sudah layak diambil review komentarnya |

**Catatan akhir:** langkah berikutnya adalah membuat batch JSON kecil untuk pencarian tempat. Analisis review dan model belum dibuat sampai daftar tempatnya disetujui.

## Tahap G - Load Raw Hasil Apify

Tahap ini membaca hasil Google Maps Search Scraper. File raw tetap disimpan apa adanya, lalu dipakai untuk membuat versi ringkas.

In [ ]:
raw_path = WORKSPACE / "01_Raw_Data" / "google_maps_search_csv" / "dataset_crawler-google-places_2026-06-09_09-19-46-159.csv"
wisata_path = WORKSPACE.parent / "Wisata_Workspace" / "01_Dataset" / "3_Curated" / "DATABASE_WISATA_LABELED_V2_REVIEWED_MEDIA_SENTIMENT_RUNTIME_CANDIDATE_2026-06-09.csv"

raw_places = pd.read_csv(raw_path)

pd.DataFrame([
    {"item": "rows", "value": len(raw_places)},
    {"item": "columns", "value": raw_places.shape[1]},
    {"item": "source_file", "value": raw_path.name},
])

**Catatan:** raw sudah ada isinya. Kolomnya sangat banyak karena detail page ikut diambil, jadi perlu dibuat versi ringkas.

## Tahap H - Ambil Kolom Inti

Dari raw scraper, yang dipakai dulu hanya kolom utama. Kolom detail lain tetap ada di raw jika nanti dibutuhkan.

In [ ]:
column_map = {
    "title": "name",
    "categoryName": "category",
    "address": "address",
    "location/lat": "latitude",
    "location/lng": "longitude",
    "totalScore": "rating",
    "reviewsCount": "reviews",
    "url": "url",
    "website": "website",
    "phone": "phone",
    "searchString": "search_query",
    "placeId": "place_id",
    "cid": "cid",
    "imageUrl": "image_url",
    "temporarilyClosed": "temporarily_closed",
    "permanentlyClosed": "permanently_closed",
}

place_core = raw_places[list(column_map.keys())].rename(columns=column_map).copy()
place_core.insert(0, "oleh_oleh_raw_id", [f"OORAW-20260609-{i:04d}" for i in range(1, len(place_core) + 1)])

coverage = place_core.replace("", pd.NA).notna().sum().reset_index()
coverage.columns = ["column", "filled_rows"]
coverage["total_rows"] = len(place_core)
coverage

**Catatan:** kolom inti sudah cukup untuk audit awal: nama, alamat, koordinat, rating, review, URL, dan query asal tersedia.

## Tahap I - Label Awal

Label ini masih sementara. Tujuannya hanya memisahkan kandidat yang terlihat kuat, data abu-abu, dan tempat yang sebaiknya ditahan dulu.

In [ ]:
import re

product_pattern = re.compile(r"oleh|bolu|brownies|bronis|kue|roti|snack|cemilan|camilan|keripik|batagor|siomay|tahu|susu|kartika|amanda|prima|rasa|raos|donat|almond|crispy|cheese|kurma|bumbu|sambal", re.I)
store_pattern = re.compile(r"toko|produsen|pemasok|pusat|oleh|kue|roti|makanan|olahan susu|tahu|suvenir|souvenir|donat|pasar", re.I)
noise_pattern = re.compile(r"taman|tujuan wisata|pusat rekreasi|taman bermain|hotel|vila|villa|penginapan|pasar apung|kafe|kedai kopi|restoran indonesia|restoran sunda|restoran$|pusat perbelanjaan", re.I)
critical_noise_pattern = re.compile(r"taman|tujuan wisata|pusat rekreasi|taman bermain|hotel|vila|villa|penginapan|pasar apung", re.I)

def label_oleh_oleh(row):
    name = str(row.get("name", ""))
    category = str(row.get("category", ""))
    text = f"{name} {category}"
    closed = str(row.get("temporarily_closed", "")).lower() == "true" or str(row.get("permanently_closed", "")).lower() == "true"
    has_product = bool(product_pattern.search(text))
    has_store = bool(store_pattern.search(text))
    has_noise = bool(noise_pattern.search(category))
    has_critical_noise = bool(critical_noise_pattern.search(category))

    if closed:
        return "held_noise", "tempat terlihat tutup sementara/permanen dari raw scraper", has_product, has_store, has_noise
    if has_critical_noise and not has_product:
        return "held_noise", "kategori lebih cocok wisata/tempat umum", has_product, has_store, has_noise
    if has_noise and not has_product:
        return "held_noise", "kategori umum tanpa sinyal produk oleh-oleh", has_product, has_store, has_noise
    if has_noise and has_product and not has_store:
        return "needs_review", "ada produk, tapi kategori masih restoran/mall/kafe", has_product, has_store, has_noise
    if has_product and has_store:
        return "candidate", "nama/kategori punya sinyal produk atau toko oleh-oleh", has_product, has_store, has_noise
    if has_product:
        return "needs_review", "ada sinyal produk, tapi kategori belum kuat", has_product, has_store, has_noise
    return "needs_review", "sinyal belum cukup kuat", has_product, has_store, has_noise

labels = place_core.apply(label_oleh_oleh, axis=1, result_type="expand")
labels.columns = ["initial_status", "hold_reason", "has_product_signal", "has_store_signal", "has_noise_category"]
place_core = pd.concat([place_core, labels], axis=1)

place_core["initial_status"].value_counts().rename_axis("initial_status").reset_index(name="rows")

**Catatan:** label awal belum dipakai untuk review scraping. Data `candidate` pun tetap perlu dicek lagi sebelum jadi target review.

## Tahap J - Cek Overlap Dengan Wisata

Tempat yang sangat mirip dengan dataset wisata ditahan dulu. Data seperti ini tidak dihapus, hanya tidak masuk target review oleh-oleh.

In [ ]:
def normalize_name(value):
    text = str(value).lower().replace("&", " dan ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\b(official|resmi|cabang|toko)\b", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def token_set(value):
    return set(token for token in normalize_name(value).split() if len(token) >= 3)

def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

wisata = pd.read_csv(wisata_path)
wisata_active = wisata.loc[wisata.get("display_status", "") != "exclude_scope", ["location_id", "location_name"]].dropna()
wisata_active["norm"] = wisata_active["location_name"].map(normalize_name)
wisata_active["tokens"] = wisata_active["location_name"].map(token_set)

def best_wisata_match(name):
    name_norm = normalize_name(name)
    name_tokens = token_set(name)
    best = ("", "", 0.0)
    for row in wisata_active.itertuples(index=False):
        score = 1.0 if name_norm == row.norm and name_norm else jaccard(name_tokens, row.tokens)
        if score > best[2]:
            best = (row.location_id, row.location_name, score)
    return best

matches = place_core["name"].apply(best_wisata_match)
place_core["wisata_overlap_id"] = matches.map(lambda x: x[0])
place_core["wisata_overlap_name"] = matches.map(lambda x: x[1])
place_core["wisata_overlap_score"] = matches.map(lambda x: round(x[2], 3))

overlap_mask = (place_core["wisata_overlap_score"] >= 0.82) & (place_core["name"].map(lambda x: len(token_set(x))) >= 2)
place_core.loc[overlap_mask, "initial_status"] = "held_wisata_overlap"
place_core.loc[overlap_mask, "hold_reason"] = "nama sangat mirip dengan dataset wisata aktif"

place_core.loc[overlap_mask, ["name", "category", "wisata_overlap_name", "wisata_overlap_score", "hold_reason"]]

**Catatan:** overlap wisata ditahan dulu. Review tempat wisata biasanya membahas pengalaman wisata, bukan khusus produk oleh-oleh.

## Tahap K - Simpan Candidate dan Manual Review

File candidate dan needs review dibuat terpisah. Candidate belum berarti final, hanya lebih siap untuk dicek tahap berikutnya.

In [ ]:
out_dir = WORKSPACE / "03_Curated"
out_dir.mkdir(parents=True, exist_ok=True)

place_core["manual_decision"] = ""
place_core["manual_note"] = ""

all_path = out_dir / "OLEH_OLEH_PLACE_DISCOVERY_CLEANED_ALL_2026-06-09.csv"
candidate_path = out_dir / "OLEH_OLEH_PLACE_DISCOVERY_CLEANED_CANDIDATE_2026-06-09.csv"
needs_review_path = out_dir / "OLEH_OLEH_PLACE_DISCOVERY_NEEDS_REVIEW_2026-06-09.csv"

place_core.to_csv(all_path, index=False)
place_core.loc[place_core["initial_status"] == "candidate"].to_csv(candidate_path, index=False)
place_core.loc[place_core["initial_status"] == "needs_review"].to_csv(needs_review_path, index=False)

pd.DataFrame([
    {"file": all_path.name, "rows": len(place_core)},
    {"file": candidate_path.name, "rows": int((place_core["initial_status"] == "candidate").sum())},
    {"file": needs_review_path.name, "rows": int((place_core["initial_status"] == "needs_review").sum())},
])

**Catatan:** file `needs_review` menjadi antrean cek manual. Data di file ini belum masuk scraping review.

## Tahap L - Simpan Data Yang Ditahan

Data yang mirip wisata, tempat umum, atau noise disimpan terpisah. Prinsipnya ditahan dulu, bukan dihapus.

In [ ]:
held_path = out_dir / "OLEH_OLEH_PLACE_DISCOVERY_HELD_NOISE_2026-06-09.csv"
held_data = place_core.loc[place_core["initial_status"].str.startswith("held")].copy()
held_data.to_csv(held_path, index=False)

held_summary = held_data["initial_status"].value_counts().rename_axis("held_status").reset_index(name="rows")
held_summary

**Catatan akhir tahap ini:** review scraping belum dijalankan. Langkah berikutnya adalah cek `candidate` dan `needs_review`, lalu pilih mana yang benar-benar layak jadi target review oleh-oleh.

## Tahap M - Load Kandidat Awal

Tahap ini membaca 84 kandidat yang sebelumnya lolos dari filter awal. Data ini belum final.

In [ ]:
candidate_path = out_dir / "OLEH_OLEH_PLACE_DISCOVERY_CLEANED_CANDIDATE_2026-06-09.csv"
candidate_places = pd.read_csv(candidate_path)

pd.DataFrame([
    {"item": "candidate_rows", "value": len(candidate_places)},
    {"item": "columns", "value": candidate_places.shape[1]},
])

**Catatan:** kandidat awal sudah cukup untuk diaudit otomatis. Hasil audit ini tetap draft, bukan keputusan akhir.

## Tahap N - Mapping Subtype Otomatis

Subtype dibuat dari nama, kategori, dan query asal. Aturan ini sederhana dan masih bisa dikoreksi manual.

In [ ]:
def auto_subtype(row):
    text = f"{row.get('name', '')} {row.get('category', '')} {row.get('search_query', '')}".lower()
    if re.search(r"brownies|bronis", text):
        return "brownies"
    if re.search(r"bolu|cake|kue balok|kue", text):
        return "bolu_cake"
    if re.search(r"roti|pastry|donat|bakery", text):
        return "pastry_roti"
    if re.search(r"snack|cemilan|camilan|keripik|crispy|almond", text):
        return "snack_kering"
    if re.search(r"tahu|susu|olahan susu|tahu susu|tauhid", text):
        return "tahu_susu"
    if re.search(r"batagor|siomay|baso tahu", text):
        return "batagor_siomay"
    if re.search(r"kopi|teh|coffee|tea", text):
        return "coffee_tea"
    if re.search(r"kurma|sambal|bumbu|abon", text):
        return "sambal_bumbu"
    if re.search(r"souvenir|suvenir|kaos|merch", text):
        return "souvenir_merch"
    if re.search(r"pusat oleh|toko oleh|oleh oleh|oleh-oleh", text):
        return "pusat_oleh_oleh"
    return "unclear"

candidate_places["auto_subtype"] = candidate_places.apply(auto_subtype, axis=1)
candidate_places["auto_subtype"].value_counts().rename_axis("auto_subtype").reset_index(name="rows")

**Catatan:** subtype paling banyak muncul dari bolu/kue, tahu/susu, snack, dan pusat oleh-oleh. Ini sesuai arah batch awal.

## Tahap O - Cek Kelengkapan Data

Cek ini melihat rating, jumlah review, koordinat, dan URL. Data yang kurang lengkap tidak langsung dipakai untuk review scraping.

In [ ]:
candidate_places["reviews_numeric"] = pd.to_numeric(candidate_places["reviews"], errors="coerce").fillna(0).astype(int)
candidate_places["rating_numeric"] = pd.to_numeric(candidate_places["rating"], errors="coerce")
candidate_places["has_coordinate"] = candidate_places[["latitude", "longitude"]].notna().all(axis=1)
candidate_places["has_url"] = candidate_places["url"].notna() & (candidate_places["url"].astype(str).str.len() > 0)
candidate_places["has_rating"] = candidate_places["rating_numeric"].between(0, 5)
candidate_places["has_reviews"] = candidate_places["reviews_numeric"] > 0
candidate_places["review_confidence_label"] = pd.cut(
    candidate_places["reviews_numeric"],
    bins=[-1, 0, 49, 499, 10**9],
    labels=["none", "low", "medium", "high"],
)

coverage_cols = ["has_coordinate", "has_url", "has_rating", "has_reviews"]
candidate_places[coverage_cols].sum().reset_index().rename(columns={"index": "check", 0: "passed_rows"})

**Catatan:** kelengkapan dasar dipakai sebagai pagar awal. Tempat dengan review rendah tetap bisa disimpan, tapi belum tentu siap discrape.

## Tahap P - Cek Duplikasi Ringan

Duplikasi di sini bukan selalu data salah. Bisa saja cabang brand yang berbeda, jadi statusnya diarahkan ke manual check.

In [ ]:
def duplicate_key(name, category):
    text = normalize_name(name)
    text = re.sub(r"\b(toko|resmi|official|cabang|oleh|khas|bandung|lembang|pusat|store|outlet|no|jl|jalan)\b", " ", text)
    tokens = [token for token in text.split() if len(token) >= 4]
    if not tokens:
        tokens = [token for token in normalize_name(category).split() if len(token) >= 4]
    return "_".join(tokens[:3])

candidate_places["duplicate_group_key"] = candidate_places.apply(lambda row: duplicate_key(row["name"], row["category"]), axis=1)
candidate_places["duplicate_count"] = candidate_places.groupby("duplicate_group_key")["duplicate_group_key"].transform("count")

candidate_places.loc[candidate_places["duplicate_count"] > 1, ["duplicate_group_key", "duplicate_count", "name", "category"]].sort_values(["duplicate_group_key", "name"]).head(30)

**Catatan:** grup yang mirip ditahan untuk dicek. Cabang brand tidak digabung otomatis.

## Tahap Q - Skor Kualitas Otomatis

Skor dibuat agar proses sortir lebih mudah. Angka ini bukan nilai bisnis final, hanya alat bantu audit.

In [ ]:
noise_category_pattern = re.compile(r"Restoran|Kafe|Kedai Kopi|Pusat Perbelanjaan|Pasar|Pujasera", re.I)
candidate_places["category_noise_signal"] = candidate_places["category"].fillna("").map(lambda x: bool(noise_category_pattern.search(str(x))))

def quality_score(row):
    score = 0
    if row["auto_subtype"] != "unclear":
        score += 25
    if str(row.get("has_product_signal", "")).lower() == "true":
        score += 15
    if str(row.get("has_store_signal", "")).lower() == "true":
        score += 15
    if row["has_coordinate"]:
        score += 10
    if row["has_url"]:
        score += 10
    if row["has_rating"]:
        score += 10
    if row["review_confidence_label"] == "high":
        score += 15
    elif row["review_confidence_label"] == "medium":
        score += 10
    elif row["review_confidence_label"] == "low":
        score += 5
    if row["category_noise_signal"]:
        score -= 15
    return max(0, min(100, score))

candidate_places["auto_quality_score"] = candidate_places.apply(quality_score, axis=1)
candidate_places[["name", "category", "auto_subtype", "reviews_numeric", "auto_quality_score"]].sort_values("auto_quality_score", ascending=False).head(15)

**Catatan:** skor tinggi membantu memilih batch uji. Skor rendah tidak dibuang, hanya belum diprioritaskan.

## Tahap R - Status Audit Otomatis

Status ini membagi data menjadi siap draft, perlu cek manual, dan tahan dulu.

In [ ]:
def auto_audit_status(row):
    if not row["has_coordinate"] or not row["has_url"]:
        return "hold_not_ready", "koordinat atau URL belum lengkap"
    if row["auto_subtype"] == "unclear":
        return "manual_check", "subtype belum jelas"
    if row["category_noise_signal"]:
        return "manual_check", "kategori masih restoran/mall/pujasera/pasar"
    if row["auto_quality_score"] >= 75 and row["reviews_numeric"] >= 20:
        return "ready_for_review_scraping", "sinyal produk/toko cukup kuat dan data utama lengkap"
    if row["auto_quality_score"] < 55 or row["reviews_numeric"] == 0:
        return "hold_not_ready", "confidence data masih rendah"
    return "manual_check", "perlu cek ringan sebelum review scraping"

status_reason = candidate_places.apply(auto_audit_status, axis=1, result_type="expand")
status_reason.columns = ["auto_audit_status", "auto_audit_reason"]
candidate_places = pd.concat([candidate_places, status_reason], axis=1)

dup_mask = (candidate_places["auto_audit_status"] == "ready_for_review_scraping") & (candidate_places["duplicate_count"] > 1)
candidate_places.loc[dup_mask, "auto_audit_status"] = "manual_check"
candidate_places.loc[dup_mask, "auto_audit_reason"] = "ada kandidat mirip/cabang, cek sebelum review scraping"

candidate_places["auto_audit_status"].value_counts().rename_axis("auto_audit_status").reset_index(name="rows")

**Catatan:** data `ready` masih draft. Untuk scraping review, sebaiknya mulai dari batch kecil dulu.

## Tahap S - Simpan Hasil Audit Otomatis

Hasil audit otomatis disimpan terpisah supaya tahap berikutnya lebih jelas.

In [ ]:
audit_cols = [
    "oleh_oleh_raw_id", "name", "category", "address", "latitude", "longitude", "rating", "reviews",
    "url", "website", "phone", "search_query", "place_id", "cid", "image_url",
    "auto_subtype", "review_confidence_label", "auto_quality_score", "duplicate_group_key", "duplicate_count",
    "category_noise_signal", "auto_audit_status", "auto_audit_reason", "manual_decision", "manual_note",
]

candidate_places["manual_decision"] = ""
candidate_places["manual_note"] = ""

auto_audit_path = out_dir / "OLEH_OLEH_CANDIDATE_AUTO_AUDIT_2026-06-09.csv"
ready_path = out_dir / "OLEH_OLEH_READY_FOR_REVIEW_SCRAPING_DRAFT_2026-06-09.csv"
manual_path = out_dir / "OLEH_OLEH_HELD_FOR_MANUAL_REVIEW_DRAFT_2026-06-09.csv"
hold_path = out_dir / "OLEH_OLEH_HOLD_NOT_READY_DRAFT_2026-06-09.csv"

candidate_places[audit_cols].to_csv(auto_audit_path, index=False)
candidate_places.loc[candidate_places["auto_audit_status"] == "ready_for_review_scraping", audit_cols].to_csv(ready_path, index=False)
candidate_places.loc[candidate_places["auto_audit_status"] == "manual_check", audit_cols].to_csv(manual_path, index=False)
candidate_places.loc[candidate_places["auto_audit_status"] == "hold_not_ready", audit_cols].to_csv(hold_path, index=False)

pd.DataFrame([
    {"file": auto_audit_path.name, "rows": len(candidate_places)},
    {"file": ready_path.name, "rows": int((candidate_places["auto_audit_status"] == "ready_for_review_scraping").sum())},
    {"file": manual_path.name, "rows": int((candidate_places["auto_audit_status"] == "manual_check").sum())},
    {"file": hold_path.name, "rows": int((candidate_places["auto_audit_status"] == "hold_not_ready").sum())},
])

**Catatan akhir audit otomatis:** hasil ini belum final. Data `manual_check` perlu dilihat singkat, sedangkan `ready_for_review_scraping` bisa dipakai untuk batch uji kecil.

## Tahap T - Load Review Test 10

Tahap ini membaca hasil review scraping uji 10 tempat. Fokusnya mengecek apakah review yang masuk cukup berguna untuk oleh-oleh.

In [ ]:
import json

review_raw_path = WORKSPACE / "05_Review_Scraping" / "raw_outputs" / "dataset_Google-Maps-Reviews-Scraper_2026-06-09_12-16-50-058.json"
with open(review_raw_path, "r", encoding="utf-8") as f:
    review_raw = pd.DataFrame(json.load(f))

pd.DataFrame([
    {"item": "review_records", "value": len(review_raw)},
    {"item": "places", "value": review_raw["title"].nunique()},
    {"item": "source_file", "value": review_raw_path.name},
])

**Catatan:** semua 10 tempat mendapat review. Tahap berikutnya hanya memakai review yang ada teks.

## Tahap U - Cleaning Review

Review tanpa teks tidak dipakai untuk analisis komentar. Review seperti itu tetap berguna sebagai rating, tapi bukan untuk membaca aspek.

In [ ]:
review_clean = review_raw.copy()
review_clean["text"] = review_clean["text"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
review_clean = review_clean.loc[review_clean["text"].str.len() > 0].copy()
review_clean.insert(0, "review_clean_id", [f"OOREV-20260609-{i:04d}" for i in range(1, len(review_clean) + 1)])
review_clean["review_usable"] = True

review_clean.groupby("title").size().reset_index(name="text_reviews").sort_values("title")

**Catatan:** review teks menjadi dasar awal. Review kosong belum dipakai untuk sentiment komentar.

## Tahap V - Aspect Flag

Flag aspek dibuat dari kata kunci sederhana. Ini belum model, hanya penanda awal supaya isi review bisa dibaca lebih cepat.

In [ ]:
aspect_patterns = {
    "aspect_product": r"donat|donut|susu|bolu|kue|roti|snack|keripik|tempe|cheese|keju|crispy|miyucheeze|balok|oleh|brownies|bronis|tahu|almond",
    "aspect_taste": r"enak|lezat|mantap|lembut|manis|gurih|asin|rasa|fresh|segar|renyah|empuk|nikmat|pas",
    "aspect_price": r"harga|murah|mahal|worth|terjangkau|promo|diskon|bayar|value",
    "aspect_packaging": r"kemasan|bungkus|box|packaging|paket|bawa|oleh|dus|kardus",
    "aspect_service": r"pelayanan|layanan|ramah|staff|staf|kasir|pegawai|cepat|lama|baik|servis|service",
    "aspect_queue": r"antri|antre|rame|ramai|penuh|sepi|nunggu|tunggu",
    "aspect_parking": r"parkir|parking",
    "aspect_tourism_context": r"wisata|foto|pemandangan|view|tempat wisata|main|jalan jalan|liburan|wahana",
}

for col, pattern in aspect_patterns.items():
    review_clean[col] = review_clean["text"].str.contains(pattern, case=False, regex=True, na=False)

aspect_cols = [col for col in aspect_patterns if col != "aspect_tourism_context"]
review_clean["aspect_count"] = review_clean[aspect_cols].sum(axis=1)

review_clean[aspect_cols + ["aspect_tourism_context"]].sum().reset_index().rename(columns={"index": "aspect", 0: "review_count"})

**Catatan:** sinyal rasa dan produk cukup terlihat. Konteks wisata rendah, jadi batch test ini masih aman untuk uji oleh-oleh.

## Tahap W - Coverage Per Tempat

Coverage dipakai untuk melihat tempat mana yang review-nya cukup membahas produk atau aspek belanja.

In [ ]:
summary_rows = []
for place, group in review_clean.groupby("title"):
    raw_count = int((review_raw["title"] == place).sum())
    text_count = len(group)
    product_count = int(group["aspect_product"].sum())
    useful_count = int((group["aspect_count"] > 0).sum())
    summary_rows.append({
        "place_name": place,
        "raw_reviews": raw_count,
        "text_reviews": text_count,
        "avg_text_review_stars": round(pd.to_numeric(group["stars"], errors="coerce").mean(), 2),
        "product_mentions": product_count,
        "taste_mentions": int(group["aspect_taste"].sum()),
        "price_mentions": int(group["aspect_price"].sum()),
        "packaging_mentions": int(group["aspect_packaging"].sum()),
        "service_mentions": int(group["aspect_service"].sum()),
        "queue_mentions": int(group["aspect_queue"].sum()),
        "parking_mentions": int(group["aspect_parking"].sum()),
        "tourism_context_mentions": int(group["aspect_tourism_context"].sum()),
        "product_mention_ratio": round(product_count / text_count, 3) if text_count else 0,
        "aspect_useful_reviews": useful_count,
        "aspect_useful_ratio": round(useful_count / text_count, 3) if text_count else 0,
    })

aspect_summary = pd.DataFrame(summary_rows).sort_values("place_name")
aspect_summary

**Catatan:** semua tempat punya review teks, tapi kekuatannya beda. Tempat dengan teks sedikit tetap bisa dipakai untuk uji, belum untuk final.

## Tahap X - Readiness Per Tempat

Readiness hanya status kerja. Status ini membantu menentukan apakah tempat cukup kuat untuk uji sentiment awal.

In [ ]:
def readiness(row):
    if row["tourism_context_mentions"] >= 5:
        return "manual_check", "ada konteks wisata cukup tinggi, cek dulu sebelum dipakai sentiment oleh-oleh"
    if row["text_reviews"] >= 20 and row["product_mention_ratio"] >= 0.45 and row["aspect_useful_ratio"] >= 0.70:
        return "strong", "review teks cukup dan banyak membahas produk/aspek"
    if row["text_reviews"] >= 12 and row["aspect_useful_ratio"] >= 0.55:
        return "medium", "cukup untuk uji awal, tapi belum kuat untuk final"
    return "weak", "review teks atau sinyal produk masih rendah"

status_reason = aspect_summary.apply(readiness, axis=1, result_type="expand")
status_reason.columns = ["sentiment_readiness", "readiness_reason"]
place_readiness = pd.concat([
    aspect_summary[["place_name", "raw_reviews", "text_reviews", "avg_text_review_stars", "product_mention_ratio", "aspect_useful_ratio", "tourism_context_mentions"]],
    status_reason,
], axis=1)

place_readiness["sentiment_readiness"].value_counts().rename_axis("sentiment_readiness").reset_index(name="places")

**Catatan:** batch test lolos sebagai data uji awal. Belum perlu scraping besar sebelum hasil cleaning ini disetujui.

## Tahap Y - Simpan Hasil Review Test

Tiga file disimpan: review cleaned, aspect summary, dan readiness tempat.

In [ ]:
review_clean_cols = [
    "review_clean_id", "title", "placeId", "cid", "categoryName", "address", "stars", "rating", "text",
    "textTranslated", "originalLanguage", "publishedAtDate", "name", "reviewerNumberOfReviews",
    "isLocalGuide", "reviewUrl", "url", "review_usable",
    "aspect_product", "aspect_taste", "aspect_price", "aspect_packaging", "aspect_service",
    "aspect_queue", "aspect_parking", "aspect_tourism_context", "aspect_count",
]

review_clean_export = review_clean[review_clean_cols].rename(columns={
    "title": "place_name",
    "placeId": "place_id",
    "categoryName": "category",
    "textTranslated": "text_translated",
    "originalLanguage": "original_language",
    "publishedAtDate": "published_at",
    "name": "reviewer_name",
    "reviewerNumberOfReviews": "reviewer_total_reviews",
    "isLocalGuide": "is_local_guide",
    "reviewUrl": "review_url",
    "url": "source_url",
})

cleaned_review_path = out_dir / "OLEH_OLEH_REVIEW_TEST10_CLEANED_2026-06-09.csv"
aspect_summary_path = out_dir / "OLEH_OLEH_REVIEW_TEST10_ASPECT_SUMMARY_2026-06-09.csv"
readiness_path = out_dir / "OLEH_OLEH_REVIEW_TEST10_PLACE_READINESS_2026-06-09.csv"

review_clean_export.to_csv(cleaned_review_path, index=False)
aspect_summary.to_csv(aspect_summary_path, index=False)
place_readiness.to_csv(readiness_path, index=False)

pd.DataFrame([
    {"file": cleaned_review_path.name, "rows": len(review_clean_export)},
    {"file": aspect_summary_path.name, "rows": len(aspect_summary)},
    {"file": readiness_path.name, "rows": len(place_readiness)},
])

**Catatan:** file cleaned hanya berisi review yang ada teks. Review tanpa teks tidak masuk file ini.

## Tahap Z - Catatan Sementara

Batch test sudah cukup untuk membuktikan jalur review bekerja. Langkah berikutnya adalah mengecek ringkasan readiness, lalu memutuskan apakah scraping dilanjutkan ke batch berikutnya.

## Tahap AA - Load Review Batch Kedua

Batch kedua berisi sisa ready draft yang aman. Data yang terkait Floating Market tidak dimasukkan dulu.

In [ ]:
review_batch2_path = WORKSPACE / "05_Review_Scraping" / "raw_outputs" / "dataset_Google-Maps-Reviews-Scraper_2026-06-09_13-42-53-867.json"
with open(review_batch2_path, "r", encoding="utf-8") as f:
    review_batch2_raw = pd.DataFrame(json.load(f))

pd.DataFrame([
    {"item": "review_records", "value": len(review_batch2_raw)},
    {"item": "places", "value": review_batch2_raw["title"].nunique()},
    {"item": "source_file", "value": review_batch2_path.name},
])

**Catatan:** semua tempat batch kedua berhasil masuk, walau beberapa tempat punya total review di bawah 30.

## Tahap AB - Cleaning Review Batch Kedua

Sama seperti batch pertama, hanya review yang punya teks yang dipakai untuk analisis komentar.

In [ ]:
review_batch2_clean = review_batch2_raw.copy()
review_batch2_clean["text"] = review_batch2_clean["text"].fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
review_batch2_clean = review_batch2_clean.loc[review_batch2_clean["text"].str.len() > 0].copy()
review_batch2_clean.insert(0, "review_clean_id", [f"OOREV2-20260609-{i:04d}" for i in range(1, len(review_batch2_clean) + 1)])
review_batch2_clean["review_batch"] = "batch2_remaining12"
review_batch2_clean["review_usable"] = True

review_batch2_clean.groupby("title").size().reset_index(name="text_reviews").sort_values("title")

**Catatan:** batch kedua punya review teks lebih sedikit dibanding raw total. Ini normal karena sebagian review hanya bintang.

## Tahap AC - Aspect Flag Batch Kedua

Pola aspek dibuat sama dengan batch pertama supaya hasilnya bisa digabung.

In [ ]:
aspect_patterns_batch2 = aspect_patterns.copy()
aspect_patterns_batch2["aspect_product"] = aspect_patterns_batch2["aspect_product"] + r"|kurma|haji|umroh|hampers|kaos|baju"
aspect_patterns_batch2["aspect_packaging"] = aspect_patterns_batch2["aspect_packaging"] + r"|hampers"

for col, pattern in aspect_patterns_batch2.items():
    review_batch2_clean[col] = review_batch2_clean["text"].str.contains(pattern, case=False, regex=True, na=False)

aspect_cols_batch2 = [col for col in aspect_patterns_batch2 if col != "aspect_tourism_context"]
review_batch2_clean["aspect_count"] = review_batch2_clean[aspect_cols_batch2].sum(axis=1)

review_batch2_clean[aspect_cols_batch2 + ["aspect_tourism_context"]].sum().reset_index().rename(columns={"index": "aspect", 0: "review_count"})

**Catatan:** batch kedua tetap relevan, tetapi kekuatannya tidak setinggi batch test pertama.

## Tahap AD - Coverage Batch Kedua

Coverage dihitung per tempat untuk melihat tempat yang cukup kuat dan yang masih lemah.

In [ ]:
def build_aspect_summary(clean_df, raw_df, batch_name):
    rows = []
    for place, group in clean_df.groupby("title"):
        raw_count = int((raw_df["title"] == place).sum())
        text_count = len(group)
        product_count = int(group["aspect_product"].sum())
        useful_count = int((group["aspect_count"] > 0).sum())
        rows.append({
            "review_batch": batch_name,
            "place_name": place,
            "raw_reviews": raw_count,
            "text_reviews": text_count,
            "avg_text_review_stars": round(pd.to_numeric(group["stars"], errors="coerce").mean(), 2),
            "product_mentions": product_count,
            "taste_mentions": int(group["aspect_taste"].sum()),
            "price_mentions": int(group["aspect_price"].sum()),
            "packaging_mentions": int(group["aspect_packaging"].sum()),
            "service_mentions": int(group["aspect_service"].sum()),
            "queue_mentions": int(group["aspect_queue"].sum()),
            "parking_mentions": int(group["aspect_parking"].sum()),
            "tourism_context_mentions": int(group["aspect_tourism_context"].sum()),
            "product_mention_ratio": round(product_count / text_count, 3) if text_count else 0,
            "aspect_useful_reviews": useful_count,
            "aspect_useful_ratio": round(useful_count / text_count, 3) if text_count else 0,
        })
    return pd.DataFrame(rows).sort_values("place_name")

batch2_aspect_summary = build_aspect_summary(review_batch2_clean, review_batch2_raw, "batch2_remaining12")
batch2_aspect_summary

**Catatan:** beberapa tempat batch kedua punya review teks di bawah 12. Tempat seperti itu belum kuat untuk final.

## Tahap AE - Readiness Batch Kedua

Readiness batch kedua dipakai untuk membedakan tempat yang cukup dipakai uji awal dan yang masih lemah.

In [ ]:
def build_readiness(summary_df):
    rows = []
    for row in summary_df.to_dict("records"):
        if row["tourism_context_mentions"] >= 5:
            status, reason = "manual_check", "ada konteks wisata cukup tinggi, cek dulu sebelum dipakai sentiment oleh-oleh"
        elif row["text_reviews"] >= 20 and row["product_mention_ratio"] >= 0.45 and row["aspect_useful_ratio"] >= 0.70:
            status, reason = "strong", "review teks cukup dan banyak membahas produk/aspek"
        elif row["text_reviews"] >= 12 and row["aspect_useful_ratio"] >= 0.55:
            status, reason = "medium", "cukup untuk uji awal, tapi belum kuat untuk final"
        else:
            status, reason = "weak", "review teks atau sinyal produk masih rendah"
        rows.append({
            "review_batch": row.get("review_batch", ""),
            "place_name": row["place_name"],
            "raw_reviews": row["raw_reviews"],
            "text_reviews": row["text_reviews"],
            "avg_text_review_stars": row["avg_text_review_stars"],
            "product_mention_ratio": row["product_mention_ratio"],
            "aspect_useful_ratio": row["aspect_useful_ratio"],
            "tourism_context_mentions": row["tourism_context_mentions"],
            "sentiment_readiness": status,
            "readiness_reason": reason,
        })
    return pd.DataFrame(rows)

batch2_readiness = build_readiness(batch2_aspect_summary)
batch2_readiness["sentiment_readiness"].value_counts().rename_axis("sentiment_readiness").reset_index(name="places")

**Catatan:** batch kedua bisa dipakai untuk memperluas data uji, tapi belum bisa disebut kuat semua.

## Tahap AF - Simpan Batch Kedua

Hasil batch kedua disimpan terpisah dulu sebelum digabung.

In [ ]:
batch2_cols = review_clean_cols.copy()

batch2_export = review_batch2_clean[batch2_cols].rename(columns={
    "title": "place_name",
    "placeId": "place_id",
    "categoryName": "category",
    "textTranslated": "text_translated",
    "originalLanguage": "original_language",
    "publishedAtDate": "published_at",
    "name": "reviewer_name",
    "reviewerNumberOfReviews": "reviewer_total_reviews",
    "isLocalGuide": "is_local_guide",
    "reviewUrl": "review_url",
    "url": "source_url",
})
batch2_export["review_batch"] = "batch2_remaining12"

batch2_cleaned_path = out_dir / "OLEH_OLEH_REVIEW_BATCH2_REMAINING12_CLEANED_2026-06-09.csv"
batch2_summary_path = out_dir / "OLEH_OLEH_REVIEW_BATCH2_REMAINING12_ASPECT_SUMMARY_2026-06-09.csv"
batch2_readiness_path = out_dir / "OLEH_OLEH_REVIEW_BATCH2_REMAINING12_PLACE_READINESS_2026-06-09.csv"

batch2_export.to_csv(batch2_cleaned_path, index=False)
batch2_aspect_summary.to_csv(batch2_summary_path, index=False)
batch2_readiness.to_csv(batch2_readiness_path, index=False)

pd.DataFrame([
    {"file": batch2_cleaned_path.name, "rows": len(batch2_export)},
    {"file": batch2_summary_path.name, "rows": len(batch2_aspect_summary)},
    {"file": batch2_readiness_path.name, "rows": len(batch2_readiness)},
])

**Catatan:** batch kedua disimpan sebagai data uji tambahan. Yang weak tetap dibawa, tapi diberi label supaya tidak terlalu dipercaya.

## Tahap AG - Gabungkan Batch 1 dan Batch 2

Gabungan ini menjadi bahan uji awal untuk baseline scoring oleh-oleh.

In [ ]:
batch1_cleaned = pd.read_csv(cleaned_review_path)
batch1_cleaned["review_batch"] = "batch1_test10"
batch2_cleaned = pd.read_csv(batch2_cleaned_path)

combined_reviews = pd.concat([batch1_cleaned, batch2_cleaned], ignore_index=True, sort=False)
combined_summary = build_aspect_summary(
    combined_reviews.rename(columns={"place_name": "title"}),
    combined_reviews.rename(columns={"place_name": "title"}),
    "combined_ready22",
)
combined_readiness = build_readiness(combined_summary)

combined_cleaned_path = out_dir / "OLEH_OLEH_REVIEW_COMBINED_READY22_CLEANED_2026-06-09.csv"
combined_summary_path = out_dir / "OLEH_OLEH_REVIEW_COMBINED_READY22_ASPECT_SUMMARY_2026-06-09.csv"
combined_readiness_path = out_dir / "OLEH_OLEH_REVIEW_COMBINED_READY22_PLACE_READINESS_2026-06-09.csv"

combined_reviews.to_csv(combined_cleaned_path, index=False)
combined_summary.to_csv(combined_summary_path, index=False)
combined_readiness.to_csv(combined_readiness_path, index=False)

pd.DataFrame([
    {"item": "text_reviews", "value": len(combined_reviews)},
    {"item": "places", "value": combined_reviews["place_name"].nunique()},
    {"item": "strong_places", "value": int((combined_readiness["sentiment_readiness"] == "strong").sum())},
    {"item": "medium_places", "value": int((combined_readiness["sentiment_readiness"] == "medium").sum())},
    {"item": "weak_places", "value": int((combined_readiness["sentiment_readiness"] == "weak").sum())},
])

**Catatan akhir:** gabungan 22 tempat cukup untuk baseline awal. Untuk final, tempat dengan status weak perlu ditahan atau diberi confidence rendah.

## Tahap AH - Load Baseline Accepted

Tahap ini membaca 22 tempat yang sudah diterima sebagai baseline awal dan review teks yang sudah siap dipakai.

In [ ]:
accepted_baseline_path = out_dir / "OLEH_OLEH_BASELINE_ACCEPTED_22_2026-06-09.csv"
review_ready_path = out_dir / "OLEH_OLEH_BASELINE_REVIEW_READY_2026-06-09.csv"

accepted_baseline = pd.read_csv(accepted_baseline_path)
review_ready = pd.read_csv(review_ready_path)

pd.DataFrame([
    {"item": "accepted_places", "value": len(accepted_baseline)},
    {"item": "review_text_rows", "value": len(review_ready)},
    {"item": "review_places", "value": review_ready["place_name"].nunique()},
])

**Catatan:** baseline sudah cukup untuk scoring awal. Data weak tetap ikut, tetapi confidence-nya lebih rendah.

## Tahap AI - Aturan Baseline Scoring

Scoring ini masih rule-based. Rating dan confidence menjadi dasar, sedangkan aspek produk/rasa/harga/pelayanan/kemasan menjadi penguat.

In [ ]:
def confidence_label_score(label):
    return {"high": 100.0, "medium": 75.0, "low": 50.0}.get(str(label), 50.0)

def ratio_score(count, total):
    return round((count / total) * 100, 2) if total else 0.0

def as_bool(series):
    return series.astype(str).str.lower().eq("true")

def score_weights(category):
    profiles = {
        "souvenir_non_makanan": {"rating": 0.35, "confidence": 0.25, "product": 0.20, "taste": 0.00, "price": 0.05, "service": 0.12, "packaging": 0.03},
        "haji_umroh_kurma": {"rating": 0.30, "confidence": 0.22, "product": 0.20, "taste": 0.04, "price": 0.10, "service": 0.07, "packaging": 0.07},
        "snack_keripik": {"rating": 0.32, "confidence": 0.24, "product": 0.17, "taste": 0.10, "price": 0.07, "service": 0.04, "packaging": 0.06},
        "olahan_susu": {"rating": 0.36, "confidence": 0.26, "product": 0.16, "taste": 0.10, "price": 0.03, "service": 0.06, "packaging": 0.03},
        "tahu_tempe": {"rating": 0.36, "confidence": 0.26, "product": 0.16, "taste": 0.10, "price": 0.03, "service": 0.06, "packaging": 0.03},
    }
    return profiles.get(category, {"rating": 0.35, "confidence": 0.25, "product": 0.15, "taste": 0.10, "price": 0.05, "service": 0.06, "packaging": 0.04})

pd.DataFrame(score_weights("kue_bolu_pastry"), index=["weight_profile_default_food"]).T

**Catatan:** bobot dibuat berbeda per kategori. Souvenir tidak dihukum karena tidak punya aspek rasa, sedangkan makanan tetap memakai sinyal rasa.

## Tahap AJ - Hitung Score Baseline

Score dihitung dari rating, confidence review, dan sinyal aspek. Hasilnya masih baseline, belum model ML.

In [ ]:
scored_rows = []
review_groups = {place: group.copy() for place, group in review_ready.groupby("place_name")}

for row in accepted_baseline.to_dict("records"):
    reviews_for_place = review_groups.get(row["name"], pd.DataFrame())
    text_reviews = len(reviews_for_place)
    product_count = int(as_bool(reviews_for_place.get("aspect_product", pd.Series(dtype=str))).sum()) if text_reviews else 0
    taste_count = int(as_bool(reviews_for_place.get("aspect_taste", pd.Series(dtype=str))).sum()) if text_reviews else 0
    price_count = int(as_bool(reviews_for_place.get("aspect_price", pd.Series(dtype=str))).sum()) if text_reviews else 0
    service_count = int(as_bool(reviews_for_place.get("aspect_service", pd.Series(dtype=str))).sum()) if text_reviews else 0
    packaging_count = int(as_bool(reviews_for_place.get("aspect_packaging", pd.Series(dtype=str))).sum()) if text_reviews else 0
    queue_count = int(as_bool(reviews_for_place.get("aspect_queue", pd.Series(dtype=str))).sum()) if text_reviews else 0
    parking_count = int(as_bool(reviews_for_place.get("aspect_parking", pd.Series(dtype=str))).sum()) if text_reviews else 0
    tourism_count = int(as_bool(reviews_for_place.get("aspect_tourism_context", pd.Series(dtype=str))).sum()) if text_reviews else 0

    rating_score = round(float(row["rating"]) / 5 * 100, 2)
    text_count_score = min(100.0, round(text_reviews / 25 * 100, 2))
    review_confidence_score = round((0.65 * text_count_score) + (0.35 * confidence_label_score(row["data_confidence_label"])), 2)

    component = {
        "rating_score": rating_score,
        "review_confidence_score": review_confidence_score,
        "product_aspect_score": ratio_score(product_count, text_reviews),
        "taste_score": ratio_score(taste_count, text_reviews),
        "price_signal_score": ratio_score(price_count, text_reviews),
        "service_score": ratio_score(service_count, text_reviews),
        "packaging_score": ratio_score(packaging_count, text_reviews),
        "queue_signal_score": ratio_score(queue_count, text_reviews),
        "parking_signal_score": ratio_score(parking_count, text_reviews),
        "tourism_context_score": ratio_score(tourism_count, text_reviews),
    }
    weights = score_weights(row["manual_category"])
    final_score = round(
        component["rating_score"] * weights["rating"]
        + component["review_confidence_score"] * weights["confidence"]
        + component["product_aspect_score"] * weights["product"]
        + component["taste_score"] * weights["taste"]
        + component["price_signal_score"] * weights["price"]
        + component["service_score"] * weights["service"]
        + component["packaging_score"] * weights["packaging"],
        2,
    )
    label = "recommended_ready" if final_score >= 75 else "usable_with_caution" if final_score >= 60 else "low_confidence"
    scored_rows.append({**row, **component, "scoring_weight_profile": ";".join(f"{k}={v}" for k, v in weights.items()), "oleh_oleh_recommendation_score": final_score, "recommendation_score_label": label})

oleh_oleh_scoring = pd.DataFrame(scored_rows).sort_values("oleh_oleh_recommendation_score", ascending=False)
oleh_oleh_scoring[["name", "manual_category", "rating_score", "review_confidence_score", "product_aspect_score", "taste_score", "oleh_oleh_recommendation_score", "recommendation_score_label"]].head(10)

**Catatan:** hasil awal scoring terlalu ketat saat aspek dibuat dominan. Formula yang dipakai di sini sudah dikalibrasi: rating dan confidence jadi dasar, aspek menjadi penguat.

## Tahap AK - Simpan dan Audit Score

Hasil scoring disimpan sebagai baseline. Label score membantu membedakan rekomendasi yang siap dan yang masih perlu hati-hati.

In [ ]:
scoring_path = out_dir / "OLEH_OLEH_BASELINE_SCORING_2026-06-09.csv"
scoring_summary_path = out_dir / "OLEH_OLEH_BASELINE_SCORING_AUDIT_SUMMARY_2026-06-09.json"

oleh_oleh_scoring.to_csv(scoring_path, index=False)

scoring_summary = {
    "generated_at": "2026-06-09",
    "total_places": int(len(oleh_oleh_scoring)),
    "total_review_texts": int(len(review_ready)),
    "score_formula_type": "category_aware_rule_based_baseline_v2",
    "calibration_note": "V1 was too strict because aspect mentions dominated. V2 prioritizes rating and review confidence, with aspects as supporting signals.",
    "status_counts": oleh_oleh_scoring["recommendation_score_label"].value_counts().to_dict(),
    "category_counts": oleh_oleh_scoring["manual_category"].value_counts().to_dict(),
}
with open(scoring_summary_path, "w", encoding="utf-8") as f:
    json.dump(scoring_summary, f, ensure_ascii=False, indent=2)

oleh_oleh_scoring["recommendation_score_label"].value_counts().rename_axis("score_label").reset_index(name="places")

**Catatan:** mayoritas tempat masuk `usable_with_caution`. Ini wajar karena data masih baseline dan beberapa tempat punya review teks terbatas.

## Tahap AL - Testing Query Baseline

Testing ini mengecek apakah ranking per kebutuhan user sudah masuk akal sebelum dipasang ke app.

In [ ]:
test_queries = [
    {"query_id": "OO-Q01", "query": "oleh-oleh kue bolu pastry Lembang", "category": "kue_bolu_pastry", "travel": "any"},
    {"query_id": "OO-Q02", "query": "oleh-oleh susu segar Lembang", "category": "olahan_susu", "travel": "any"},
    {"query_id": "OO-Q03", "query": "snack kering tahan perjalanan", "category": "snack_keripik", "travel": "high"},
    {"query_id": "OO-Q04", "query": "oleh-oleh tahu tempe", "category": "tahu_tempe", "travel": "any"},
    {"query_id": "OO-Q05", "query": "souvenir kaos Bandung", "category": "souvenir_non_makanan", "travel": "any"},
    {"query_id": "OO-Q06", "query": "oleh-oleh haji umroh kurma", "category": "haji_umroh_kurma", "travel": "any"},
    {"query_id": "OO-Q07", "query": "rekomendasi oleh-oleh terbaik semua kategori", "category": "all", "travel": "any"},
    {"query_id": "OO-Q08", "query": "oleh-oleh yang aman dibawa perjalanan jauh", "category": "all", "travel": "high"},
]

test_rows = []
for query in test_queries:
    candidates = oleh_oleh_scoring.copy()
    if query["category"] != "all":
        candidates = candidates.loc[candidates["manual_category"] == query["category"]].copy()
    candidates["query_adjusted_score"] = candidates["oleh_oleh_recommendation_score"]
    candidates["reason"] = "base_score"
    if query["travel"] == "high":
        candidates.loc[candidates["travel_durability"] == "high", "query_adjusted_score"] += 5
        candidates.loc[candidates["travel_durability"] == "high", "reason"] = "travel_high_boost"
        candidates.loc[candidates["travel_durability"] == "low", "query_adjusted_score"] -= 5
        candidates.loc[candidates["travel_durability"] == "low", "reason"] = "travel_low_penalty"
    for rank, result in enumerate(candidates.sort_values("query_adjusted_score", ascending=False).head(5).to_dict("records"), start=1):
        test_rows.append({
            "query_id": query["query_id"],
            "query": query["query"],
            "rank": rank,
            "name": result["name"],
            "manual_category": result["manual_category"],
            "query_adjusted_score": round(result["query_adjusted_score"], 2),
            "recommendation_score_label": result["recommendation_score_label"],
            "reason": result["reason"],
        })

scoring_test = pd.DataFrame(test_rows)
test_path = out_dir / "OLEH_OLEH_BASELINE_SCORING_TEST_QUERIES_2026-06-09.csv"
scoring_test.to_csv(test_path, index=False)
scoring_test.groupby("query_id").head(3)[["query_id", "rank", "name", "manual_category", "query_adjusted_score"]]

**Catatan:** hasil testing sudah masuk akal untuk baseline. Query kategori mengembalikan kategori yang sesuai, dan query perjalanan jauh memberi dorongan ke item yang lebih tahan dibawa.

## Tahap AM - Keputusan Sementara Scoring

Baseline scoring diterima untuk uji awal. Hasil ini belum model final dan belum menggantikan audit manual, tetapi sudah cukup untuk mulai menguji fitur rekomendasi oleh-oleh.

## Tahap AN - Apply Manual Product and Price

Manual completion produk dan harga oleh-oleh sudah diisi untuk 22 data.

In [ ]:
import re
import json
from pathlib import Path

filled_path = root / "OLEH_OLEH_MANUAL_COMPLETION_PRICE_PRODUCT_FILLED_2026-06-10.csv"
enriched_path = out_dir / "OLEH_OLEH_BASELINE_UI_ENRICHED_2026-06-10.csv"
final_path = out_dir / "OLEH_OLEH_BASELINE_UI_ENRICHED_WITH_MANUAL_PRODUCT_PRICE_2026-06-10.csv"
final_summary_path = out_dir / "OLEH_OLEH_BASELINE_UI_ENRICHED_WITH_MANUAL_PRODUCT_PRICE_SUMMARY_2026-06-10.json"

manual_filled = pd.read_csv(filled_path)
ui_enriched = pd.read_csv(enriched_path)

def normalize_confidence(value):
    text = str(value).strip().lower()
    return {"tinggi": "high", "sedang": "medium", "rendah": "low"}.get(text, text or "unknown")

def normalize_status(value):
    text = str(value).strip().lower()
    return {"completed": "complete", "selesai": "complete"}.get(text, text or "unknown")

def normalize_durability(value):
    text = str(value).strip().lower()
    if re.search(r"bulan|tahun|sangat lama|berbulan", text):
        return "tinggi"
    if re.search(r"kulkas|dingin|3-4 hari|3-5 hari|4-5 hari", text):
        return "sedang"
    if re.search(r"1-2 hari|1-3 hari|2-3 hari|mentah", text):
        return "rendah"
    return "unknown"

def parse_price(value):
    text = str(value)
    numbers = [int(match.group(0).replace(".", "")) for match in re.finditer(r"\d[\d\.]*", text)]
    price_min = numbers[0] if numbers else None
    price_max = numbers[1] if len(numbers) > 1 else price_min
    if price_min is None:
        bucket = "unknown"
    elif price_min < 50000:
        bucket = "<50k"
    elif price_min < 100000:
        bucket = "50k-100k"
    elif price_min < 200000:
        bucket = "100k-200k"
    else:
        bucket = "200k+"
    return pd.Series({"price_min_idr": price_min, "price_max_idr": price_max, "price_bucket": bucket, "price_has_plus": "+" in text})

manual_cols = [
    "oleh_oleh_id", "produk_utama_manual", "price_range_manual", "price_confidence_manual",
    "daya_tahan_produk_manual", "cocok_untuk_manual", "best_use_case_manual",
    "catatan_manual", "evidence_source_manual", "manual_status",
]
manual_ready = manual_filled[manual_cols].copy()
price_parts = manual_ready["price_range_manual"].apply(parse_price)
manual_ready = pd.concat([manual_ready, price_parts], axis=1)
manual_ready["price_confidence"] = manual_ready["price_confidence_manual"].apply(normalize_confidence)
manual_ready["daya_tahan_produk_class"] = manual_ready["daya_tahan_produk_manual"].apply(normalize_durability)
manual_ready["manual_product_price_status"] = manual_ready["manual_status"].apply(normalize_status)

final_dataset = ui_enriched.merge(manual_ready, on="oleh_oleh_id", how="left")
final_dataset = final_dataset.rename(columns={
    "produk_utama_manual": "produk_utama",
    "price_range_manual": "price_range_label",
    "daya_tahan_produk_manual": "daya_tahan_produk_label",
    "cocok_untuk_manual": "cocok_untuk",
    "best_use_case_manual": "best_use_case",
    "catatan_manual": "manual_product_price_note",
    "evidence_source_manual": "manual_product_price_evidence",
})

final_dataset.to_csv(final_path, index=False)

manual_summary = {
    "generated_at": "2026-06-10",
    "rows": int(len(final_dataset)),
    "manual_rows": int(len(manual_filled)),
    "joined_rows": int(final_dataset["produk_utama"].notna().sum()),
    "price_numeric_rows": int(final_dataset["price_min_idr"].notna().sum()),
    "price_confidence_counts": final_dataset["price_confidence"].value_counts(dropna=False).to_dict(),
    "durability_class_counts": final_dataset["daya_tahan_produk_class"].value_counts(dropna=False).to_dict(),
    "output_file": str(final_path),
}
with open(final_summary_path, "w", encoding="utf-8") as f:
    json.dump(manual_summary, f, ensure_ascii=False, indent=2)

pd.DataFrame([manual_summary])[["rows", "manual_rows", "joined_rows", "price_numeric_rows", "output_file"]]

**Keputusan:** data diterima karena 22/22 lengkap. Harga dipakai sebagai estimasi manual, bukan harga resmi.